## Validando a SparkSession

In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

## Conectando Azure ADLS Gen2 no Databricks

### Mostrando os pontos de montagem no cluster Databricks

In [0]:
display(dbutils.fs.mounts())

### Definindo função para montar ADLS com SAS token

In [0]:
storageAccountName = "datalake7eadf73a479de9f7"
sasToken = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-06-24T07:28:34Z&st=2025-06-23T23:28:34Z&spr=https&sig=idw%2B%2FOotMzKptQ8X5oECtDziKZ8sgPZM9CvPHeuOhqA%3D"

def mount_adls(blobContainerName):
    mount_point = f"/mnt/{storageAccountName}/{blobContainerName}"
    
    # Verificar se já está montado
    existing_mounts = [mount.mountPoint for mount in dbutils.fs.mounts()]
    
    if mount_point in existing_mounts:
        print(f"AVISO: {mount_point} já está montado. Desmontando primeiro...")
        try:
            dbutils.fs.unmount(mount_point)
            print(f"Desmontado com sucesso: {mount_point}")
        except Exception as e:
            print(f"Erro ao desmontar {mount_point}: {e}")
            return False
    
    # Montar o container
    try:
        dbutils.fs.mount(
            source = "wasbs://{}@{}.blob.core.windows.net".format(blobContainerName, storageAccountName),
            mount_point = mount_point,
            extra_configs = {'fs.azure.sas.' + blobContainerName + '.' + storageAccountName + '.blob.core.windows.net': sasToken}
        )
        print(f"OK! Montado: {mount_point}")
        return True
    except Exception as e:
        print(f"Falha ao montar {mount_point}: {e}")
        return False

### Função para verificar se ambiente já está montado

In [0]:
def mount_adls_safe(blobContainerName):
    mount_point = f"/mnt/{storageAccountName}/{blobContainerName}"
    existing_mounts = [mount.mountPoint for mount in dbutils.fs.mounts()]
    
    if mount_point in existing_mounts:
        print(f"OK! {mount_point} já está montado.")
        return True
    else:
        return mount_adls(blobContainerName)


### Montando containers

In [0]:
mount_adls('landing-zone')
mount_adls('bronze')

AVISO: /mnt/datalake7eadf73a479de9f7/landing-zone já está montado. Desmontando primeiro...
/mnt/datalake7eadf73a479de9f7/landing-zone has been unmounted.
Desmontado com sucesso: /mnt/datalake7eadf73a479de9f7/landing-zone
OK! Montado: /mnt/datalake7eadf73a479de9f7/landing-zone
AVISO: /mnt/datalake7eadf73a479de9f7/bronze já está montado. Desmontando primeiro...
/mnt/datalake7eadf73a479de9f7/bronze has been unmounted.
Desmontado com sucesso: /mnt/datalake7eadf73a479de9f7/bronze
OK! Montado: /mnt/datalake7eadf73a479de9f7/bronze
Out[48]: True

### Limpar container da camada bronze

In [0]:
def clean_bronze_container():
    bronze_path = f"/mnt/{storageAccountName}/bronze"
    
    try:
        # Verificar se o path existe
        try:
            files_and_dirs = dbutils.fs.ls(bronze_path)
        except Exception:
            print(f"Path {bronze_path} não existe ou está vazio.")
            return True
        
        if not files_and_dirs:
            print("Container bronze já está vazio.")
            return True
        
        print(f"Encontrados {len(files_and_dirs)} itens no bronze para exclusão...")
        
        # Excluir cada item (arquivos e diretórios)
        for item in files_and_dirs:
            try:
                dbutils.fs.rm(item.path, True)  # True para recursivo
                print(f"✓ Excluído: {item.name}")
            except Exception as e:
                print(f"✗ Erro ao excluir {item.name}: {str(e)}")
                return False
        
        print("✅ Limpeza do container bronze concluída com sucesso!")
        return True
        
    except Exception as e:
        print(f"❌ Erro geral na limpeza do bronze: {str(e)}")
        return False


### Executar limpeza do diretório

In [0]:
clean_bronze_container()

Container bronze já está vazio.
Out[50]: True

### Processando todos os CSVs da landing-zone

In [0]:
def process_all_csvs():
    landing_zone_path = f"/mnt/{storageAccountName}/landing-zone"
    bronze_path = f"/mnt/{storageAccountName}/bronze"
    
    def get_all_csv_files(path):
        all_files = []
        try:
            items = dbutils.fs.ls(path)
            for item in items:
                if item.isDir():
                    all_files.extend(get_all_csv_files(item.path))
                elif item.name.lower().endswith('.csv'):
                    all_files.append(item)
        except:
            pass
        return all_files
    
    csv_files = get_all_csv_files(landing_zone_path)
    print(f"Encontrados {len(csv_files)} arquivos CSV")
    
    dataframes = {}
    
    for file in csv_files:
        try:
            # Limpar o nome da tabela - remover prefixos problemáticos
            relative_path = file.path.replace(landing_zone_path, '').lstrip('/')
            table_name = relative_path.replace('.csv', '').replace('/', '_').replace(':', '').replace('-', '_').lower()
            
            print(f"Processando: {file.name} -> {table_name}")
            
            # Tentar ler com schema automático primeiro
            try:
                df = spark.read.option("inferSchema", "true").option("header", "true").csv(file.path)
            except:
                # Se falhar, ler como string e depois converter
                print(f"  Fallback: lendo {file.name} como strings")
                df = spark.read.option("header", "true").csv(file.path)
            
            df.write.format('delta').mode('overwrite').save(f"{bronze_path}/{table_name}")
            dataframes[table_name] = df
            print(f"  ✓ Sucesso: {table_name}")
            
        except Exception as e:
            print(f"  ✗ Erro em {file.name}: {str(e)}")
    
    return dataframes

In [0]:
dataframes_processados = process_all_csvs()

Encontrados 15 arquivos CSV
Processando: _prisma_migrations_20250621_014537.csv -> dbfs_excel_exports__prisma_migrations_20250621_014537
  Fallback: lendo _prisma_migrations_20250621_014537.csv como strings
  ✗ Erro em _prisma_migrations_20250621_014537.csv: Unable to infer schema for CSV. It must be specified manually.
Processando: achievement_unlocked_20250621_014612.csv -> dbfs_excel_exports_achievement_unlocked_20250621_014612
  ✓ Sucesso: dbfs_excel_exports_achievement_unlocked_20250621_014612
Processando: achievements_20250621_014608.csv -> dbfs_excel_exports_achievements_20250621_014608
  ✓ Sucesso: dbfs_excel_exports_achievements_20250621_014608
Processando: developers_20250621_014548.csv -> dbfs_excel_exports_developers_20250621_014548
  ✓ Sucesso: dbfs_excel_exports_developers_20250621_014548
Processando: dlcs_20250621_014615.csv -> dbfs_excel_exports_dlcs_20250621_014615
  ✓ Sucesso: dbfs_excel_exports_dlcs_20250621_014615
Processando: game_genders_20250621_014619.csv -> dbf

### Criando tabelas Delta externas

In [0]:
def create_external_tables():
    bronze_path = f"/mnt/{storageAccountName}/bronze"
    
    try:
        bronze_dirs = dbutils.fs.ls(bronze_path)
        
        for dir_info in bronze_dirs:
            if dir_info.isDir():
                table_name = dir_info.name.rstrip('/')
                
                spark.sql(f"DROP TABLE IF EXISTS {table_name}")
                spark.sql(f"CREATE TABLE {table_name} USING DELTA LOCATION '{dir_info.path}'")
                print(f"Tabela externa: {table_name}")
                    
    except Exception as e:
        print(f"Erro: {str(e)}")

In [0]:
create_external_tables()

Tabela externa: dbfs_excel_exports_achievement_unlocked_20250621_014612
Tabela externa: dbfs_excel_exports_achievements_20250621_014608
Tabela externa: dbfs_excel_exports_developers_20250621_014548
Tabela externa: dbfs_excel_exports_dlcs_20250621_014615
Tabela externa: dbfs_excel_exports_game_genders_20250621_014619
Tabela externa: dbfs_excel_exports_game_platforms_20250621_014625
Tabela externa: dbfs_excel_exports_game_tags_20250621_014622
Tabela externa: dbfs_excel_exports_games_20250621_014544
Tabela externa: dbfs_excel_exports_genders_20250621_014554
Tabela externa: dbfs_excel_exports_platforms_20250621_014551
Tabela externa: dbfs_excel_exports_purchases_20250621_014605
Tabela externa: dbfs_excel_exports_reviews_20250621_014601
Tabela externa: dbfs_excel_exports_tags_20250621_014558
Tabela externa: dbfs_excel_exports_users_20250621_014541


### Criando tabelas Delta gerenciadas

In [0]:
def create_managed_tables():
    database_name = "pipeline"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")
    
    bronze_path = f"/mnt/{storageAccountName}/bronze"
    
    try:
        bronze_dirs = dbutils.fs.ls(bronze_path)
        
        for dir_info in bronze_dirs:
            if dir_info.isDir():
                table_name = dir_info.name.rstrip('/')
                
                df = spark.read.format('delta').load(dir_info.path)
                df.write.format('delta').mode('overwrite').saveAsTable(f"{database_name}.{table_name}")
                print(f"Tabela gerenciada: {database_name}.{table_name}")
                    
    except Exception as e:
        print(f"Erro: {str(e)}")

In [0]:
create_managed_tables()

Tabela gerenciada: pipeline.dbfs_excel_exports_achievement_unlocked_20250621_014612
Tabela gerenciada: pipeline.dbfs_excel_exports_achievements_20250621_014608
Tabela gerenciada: pipeline.dbfs_excel_exports_developers_20250621_014548
Tabela gerenciada: pipeline.dbfs_excel_exports_dlcs_20250621_014615
Tabela gerenciada: pipeline.dbfs_excel_exports_game_genders_20250621_014619
Tabela gerenciada: pipeline.dbfs_excel_exports_game_platforms_20250621_014625
Tabela gerenciada: pipeline.dbfs_excel_exports_game_tags_20250621_014622
Tabela gerenciada: pipeline.dbfs_excel_exports_games_20250621_014544
Tabela gerenciada: pipeline.dbfs_excel_exports_genders_20250621_014554
Tabela gerenciada: pipeline.dbfs_excel_exports_platforms_20250621_014551
Tabela gerenciada: pipeline.dbfs_excel_exports_purchases_20250621_014605
Tabela gerenciada: pipeline.dbfs_excel_exports_reviews_20250621_014601
Tabela gerenciada: pipeline.dbfs_excel_exports_tags_20250621_014558
Tabela gerenciada: pipeline.dbfs_excel_exports

### Validando tabelas criadas

In [0]:
print("=== TABELAS GERENCIADAS ===")
spark.sql("SHOW TABLES IN pipeline").show()

=== TABELAS GERENCIADAS ===
+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|pipeline|achievement_unlocked|      false|
|pipeline|        achievements|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|dbfs_excel_export...|      false|
|pipeline|          developers|      false|
|pipeline|                dlcs|      false|
|pipeline|        game_genders|      false|
|pip

In [0]:
display(dbutils.fs.ls(f"/mnt/{storageAccountName}/bronze"))

path,name,size,modificationTime
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_achievement_unlocked_20250621_014612/,dbfs_excel_exports_achievement_unlocked_20250621_014612/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_achievements_20250621_014608/,dbfs_excel_exports_achievements_20250621_014608/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_developers_20250621_014548/,dbfs_excel_exports_developers_20250621_014548/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_dlcs_20250621_014615/,dbfs_excel_exports_dlcs_20250621_014615/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_game_genders_20250621_014619/,dbfs_excel_exports_game_genders_20250621_014619/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_game_platforms_20250621_014625/,dbfs_excel_exports_game_platforms_20250621_014625/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_game_tags_20250621_014622/,dbfs_excel_exports_game_tags_20250621_014622/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_games_20250621_014544/,dbfs_excel_exports_games_20250621_014544/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_genders_20250621_014554/,dbfs_excel_exports_genders_20250621_014554/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_platforms_20250621_014551/,dbfs_excel_exports_platforms_20250621_014551/,0,0
